In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.environ.get('OPENAI_API_KEY'),model="gpt-3.5-turbo",temperature=0)
#OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.environ.get('OPENAI_API_KEY'))

## **RAG IMPLEMENTATION WITH OWN TEXT DATA** ##

#### **STEP 1: Preparing Document for our text** ####

In [4]:
from langchain_core.documents import Document

In [5]:
my_text="""Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]"""

In [6]:
docs=[Document(page_content=my_text,metadata={"source":"ABC","documentID":"Doc1"})]
docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a

#### **STEP 2: Splitting the document into CHUNKS** ####

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='The traditional goals of AI resea

#### **STEP 3: Creating Embeddings for the CHUNKS** ####

In [8]:
from langchain_openai import OpenAIEmbeddings 

In [9]:
embedding_model = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    model="openai/text-embedding-3-small"
)

#### **Step4: Create and Store Embeddings of the chunks in Vector DB** ####

In [10]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=chunks,embedding=embedding_model)

C:\Users\Krishna\AppData\Local\Temp\ipykernel_2848\1377923466.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


#### **Step5: Semantic Search** ####

In [11]:
vectors = []
for doc in chunks:
    vector = embedding_model.embed_documents([doc.page_content])
    vectors.append(vector)


In [ ]:
context=vectorstore.similarity_search('What is AI?',k=2)

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).')]

#### **Step5: Talk to LLM** ####

In [13]:
llm.invoke('What is AI? Answer using following context:{context}')

AIMessage(content='AI, or artificial intelligence, is a branch of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence. This includes tasks such as visual perception, speech recognition, decision-making, and language translation. AI systems are designed to learn from data, adapt to new situations, and improve their performance over time. AI technology is used in a wide range of applications, from virtual assistants like Siri and Alexa to self-driving cars and medical diagnosis systems.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 18, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': No